In [ ]:
import pandas as pd
from common import *
from scipy.stats import chi2_contingency

In [ ]:
main_dir = "data/home-credit-credit-risk-model-stability/parquet_files/"
main_dir_train = f"{main_dir}train/"

In [ ]:
train_base = pd.read_parquet(f"{main_dir_train}train_base.parquet")
train_base = train_base[["case_id","target"]]

# Person
person_names = ["train_person_1", "train_person_2"]
train_person = []
for person in person_names:
    train_person.append(pd.read_parquet(f"{main_dir_train}{person}.parquet"))

In [ ]:
train_person_1 = train_person[0]
train_person_2 = train_person[1]
del train_person

Train person 1

In [ ]:
train_person_1 = train_person_1[["case_id", "num_group1"] + [x for x in train_person_1.columns if x not in ["case_id", "num_group1"]]]
for col in ["birth_259D", "birthdate_87D"]:
    train_person_1[col] = pd.to_datetime(train_person_1[col], format="%Y-%m-%d")
# train_person_1 = train_person_1.replace({"a55475b1":pd.NA})
train_person_1.head()

In [ ]:
train_person_1_person = train_person_1[train_person_1["num_group1"]==0]
train_person_1_person.head()

In [ ]:
train_person_1_f = train_person_1_person[["case_id", "education_927M"]]

In [ ]:
plot_null_percent(train_person_1_person)
train_person_1_person = delete_null_columns(train_person_1_person, 0.1)
train_person_1_person = train_person_1_person.merge(train_base, on=["case_id"])
# we have education in another dataset, as well as childnum (apprlpv)
# join gender-sex, educations, birth

In [ ]:
train_person_1_person.incometype_1044T.value_counts()
train_person_1_person.contaddr_zipcode_807M.value_counts()
train_person_1_person.contaddr_smempladdr_334L.value_counts()


Categorical columns

In [ ]:
selected_categorical = []
for col in train_person_1_person.columns:
    if train_person_1_person[col].dtype in ["object","categorical"]:
        counts = train_person_1_person[col].value_counts(normalize=True).reset_index()
        plt.figure(figsize=(25,5))
        plt.subplot(1,2,1)
        sns.barplot(data=counts, x=col, y="proportion")
        counts = train_person_1_person.groupby(["target"])[col].value_counts(normalize=True).reset_index()
        plt.subplot(1,2,2)
        sns.barplot(data=counts, x=col, y="proportion", hue="target")

        contingency = pd.crosstab(index=train_person_1_person["target"],
                                  columns = [train_person_1_person[col]])
        statistic, pvalue, dof, expected_freq = chi2_contingency(contingency.values)
        plt.suptitle(f"Chi-square {col}: p-value = {pvalue}")

        if pvalue < 0.05:
            selected_categorical.append(col)
        plt.show()

Numerical columns

In [ ]:
numerical_person_cols = [x for x in train_person_1_person.columns
                         if train_person_1_person[x].dtype in ["int64","float64"] and
                            x not in ["case_id","num_group1", "target"]]
train_person_1_person[numerical_person_cols][train_person_1_person["persontype_1072L"]!=train_person_1_person["persontype_792L"]]

In [ ]:
plt.figure(figsize=(25,5))
plt.subplot(1,2,1)
sns.boxplot(data=train_person_1_person, x="mainoccupationinc_384A", hue="target")
plt.subplot(1,2,2)
sns.kdeplot(train_person_1_person, x="mainoccupationinc_384A",
            hue="target", common_norm=False)

Combination

In [ ]:
train_person_1_f = pd.merge(train_person_1_f,
                            train_person_1_person[["case_id"] + selected_categorical],
                            on=["case_id"])
train_person_1_f["incometype_1044T"] = train_person_1_f["incometype_1044T"].map(lambda x: "HANDICAPPED" if "HANDIC" in x else x)

In [ ]:
train_person_1_f = pd.merge(train_person_1_f,
                            train_person_1_person[["case_id", "mainoccupationinc_384A"]],
                            on=["case_id"])

In [ ]:
train_person_1_f.head()

habría que considerar, a la hora de crear las features del modelo, ver si dejar los contaddr (hay muchas categorías), y los registaddr. el tipo de  teléfono mayoritariamente es primary mobile pero se puede probar también

In [ ]:
features_train_person_1 = get_column_descriptions(train_person_1)
features_train_person_1

In [ ]:
# For the feature engineering part
zip_code = train_person_1_f["contaddr_zipcode_807M"].str.split("_", expand=True)
def transform_zip_code(zip: str):
    try:
        return int(zip[1:])
    except:
        return pd.NA
zip_code.loc[:, 0] = zip_code.loc[:, 0].map(transform_zip_code)
zip_code_cols = []
for col in range(len(zip_code.columns)):
    zip_code.loc[:, col] = pd.to_numeric(zip_code.loc[:, col], errors="coerce")
    zip_code_cols.append(f"zip_code_part_{col+1}")
zip_code.columns = zip_code_cols

train_person_1_f.drop(columns=[x for x in train_person_1_f.columns
                               if "cont" in x or "zipcode" in x],
                      inplace=True)
train_person_1_f = pd.concat([train_person_1_f, zip_code], axis=1)

Train person 2

In [ ]:
train_person_2.head()

In [ ]:
features_train_person_2 = get_column_descriptions(train_person_2)
features_train_person_2

In [ ]:
train_person_2 = train_person_2.replace({"a55475b1":pd.NA})
train_person_2_person = train_person_2[train_person_2["num_group1"]==0]
plot_null_percent(train_person_2_person)